# NeuroGuard — CNN vs Transformer MRI architecture study

Trains **ViT-B/16** and **DINOv2-small** (Hugging Face) plus a **ResNet-18** rerun
under the exact same subject-stratified 5-fold CV as the deployed model, on a free
Colab T4 GPU. Deployed artifacts are never touched — transformer runs write
`mri_metrics_<arch>.json` / `preds_<arch>.csv` only.

**Before running:**
1. Runtime → Change runtime type → **T4 GPU**.
2. Have `processed_v2.zip` ready (zip of `data/processed_v2/` containing
   `manifest_v2.csv` + `slices/`). Upload it in the upload cell below, or put it in
   your Google Drive as `MyDrive/processed_v2.zip` and use the Drive cell instead.
3. Run all cells top to bottom (~1 hour total). The last cell downloads the results.

In [ ]:
!nvidia-smi
!pip -q install transformers

In [ ]:
!git clone --depth 1 https://github.com/akashjacob2005-rgb/final-year-project.git
%cd final-year-project

## Data — run ONE of the next two cells

In [ ]:
# Option A: browser upload (~100-200 MB, a few minutes)
from google.colab import files
uploaded = files.upload()  # pick processed_v2.zip
!unzip -q processed_v2.zip -d /content/data
!ls /content/data/processed_v2 | head

In [ ]:
# Option B: from Google Drive (faster if already uploaded there)
from google.colab import drive
drive.mount('/content/drive')
!unzip -q /content/drive/MyDrive/processed_v2.zip -d /content/data
!ls /content/data/processed_v2 | head

## Train all three architectures (same folds, same seed, same eval)
ResNet-18 is rerun here so every number in the comparison comes from the identical
environment. Each run prints its CV summary at the end.

In [ ]:
DATA = '/content/data/processed_v2'
OUT = '/content/results'
!python ml/mri/train_mri.py --data-dir $DATA --out-dir $OUT --arch resnet18 --folds 5 --device cuda --save-preds

In [ ]:
!python ml/mri/train_mri.py --data-dir $DATA --out-dir $OUT --arch dinov2_s --folds 5 --device cuda --save-preds

In [ ]:
!python ml/mri/train_mri.py --data-dir $DATA --out-dir $OUT --arch vit_b16 --folds 5 --device cuda --save-preds

## Ensemble + download results

In [ ]:
!python ml/mri/ensemble_eval.py $OUT/preds_resnet18.csv $OUT/preds_dinov2_s.csv --out $OUT/ensemble_resnet_dinov2.json
!python ml/mri/ensemble_eval.py $OUT/preds_resnet18.csv $OUT/preds_vit_b16.csv --out $OUT/ensemble_resnet_vit.json
!python ml/mri/ensemble_eval.py $OUT/preds_resnet18.csv $OUT/preds_dinov2_s.csv $OUT/preds_vit_b16.csv --out $OUT/ensemble_all3.json

In [ ]:
# Bundle everything EXCEPT the .pt weights (big; only metrics + preds are needed
# locally) and download. Bring the zip back to ml/artifacts/ on your machine.
!cd /content/results && zip -q arch_comparison_results.zip *.json preds_*.csv
from google.colab import files
files.download('/content/results/arch_comparison_results.zip')